# StreamSense — Live Serving + Dashboard (Colab)

**Nothing is retrained or recomputed here.** The models (`models/`) and the processed data
(`data/processed/`) are already committed in the repo — this notebook only:
1. installs the serving binaries (a fresh Colab has none), then
2. **loads** the committed models and **serves** them, and launches the dashboard.

Serves: retrieval (`:8501`) + TFX ranker (`:8502`) via `tensorflow_model_server`, and the PyTorch
ranker (`:8080`) via TorchServe. Dashboard modes: **Testing** = TFX ranker; **Prod** = retrieval → PyTorch ranker.

> The only slow cell is #1 (installing software). Every other cell is fast — it just loads and serves.

## 1. Clone + install the serving binaries (one-time infra install — no model work)

In [ ]:
!git clone https://github.com/techykajal/streamsense-ott-recsys.git 2>/dev/null || true
%cd streamsense-ott-recsys
# TensorFlow Serving binary
!echo "deb http://storage.googleapis.com/tensorflow-serving-apt stable tensorflow-model-server tensorflow-model-server-universal" > /etc/apt/sources.list.d/tensorflow-serving.list
!curl -s https://storage.googleapis.com/tensorflow-serving-apt/tensorflow-serving.release.pub.gpg | apt-key add - 2>/dev/null
!apt-get update -q && apt-get install -y -q tensorflow-model-server openjdk-17-jdk-headless
# TorchServe + dashboard deps (no model libraries — we only load & serve)
!pip -q install torchserve torch-model-archiver streamlit
print("install done")

## 2. Verify the committed models + data are present (nothing is rebuilt)

In [ ]:
import os, glob
checks = {
    "data (parquet)":  "data/processed/interactions.parquet",
    "retrieval model": "models/retrieval",
    "torch ranker":    "models/ranker.pt",
    "id maps":         "models/id_maps.json",
    "TFX ranker":      "models/ranking_tf",
}
for name, p in checks.items():
    print(f"{name:16}:", "OK" if os.path.exists(p) else "MISSING")
assert os.path.exists("data/processed/interactions.parquet"), "committed data missing — check the repo"
print("TFX ranker version dir:", glob.glob("models/ranking_tf/*/"))

## 3. Serve retrieval (:8501) + TFX ranker (:8502) — loads committed SavedModels

In [ ]:
import subprocess, os, time
ROOT = os.getcwd()
os.makedirs("serve/retrieval/1", exist_ok=True)
!cp -r models/retrieval/* serve/retrieval/1/ 2>/dev/null || true    # TF Serving needs a version dir

subprocess.Popen(["tensorflow_model_server","--rest_api_port=8501",
                  "--model_name=retrieval", f"--model_base_path={ROOT}/serve/retrieval"])
subprocess.Popen(["tensorflow_model_server","--rest_api_port=8502",
                  "--model_name=ranker", f"--model_base_path={ROOT}/models/ranking_tf"])
time.sleep(8)
!curl -s http://localhost:8501/v1/models/retrieval | head -c 200
print()
!curl -s http://localhost:8502/v1/models/ranker | head -c 200

## 4. Serve the PyTorch ranker via TorchServe (:8080) — loads committed ranker.pt

In [ ]:
import os, time
os.makedirs("model_store", exist_ok=True)
!torch-model-archiver --model-name ranker --version 1.0     --serialized-file models/ranker.pt     --handler serving/torchserve_handler.py     --extra-files "src/ranking_torch.py,models/ranker.pt"     --export-path model_store -f
!torchserve --stop 2>/dev/null; sleep 2
!torchserve --start --ncs --model-store model_store --models ranker=ranker.mar --disable-token-auth 2>/dev/null
time.sleep(12)
!curl -s http://localhost:8080/ping

## 5. Smoke-test the serving APIs (fast — just HTTP calls)

In [ ]:
import requests
rows=[{"user":0,"movie":1,"seg":0},{"user":0,"movie":2,"seg":0}]
try: print("torch    :", requests.post("http://localhost:8080/predictions/ranker", json=rows, timeout=20).text[:200])
except Exception as e: print("torch err:", e)
print("retrieval:", requests.post("http://localhost:8501/v1/models/retrieval:predict",
      json={"inputs":{"user_id":["1"],"segment":[0]}}, timeout=20).text[:200])
import base64, tensorflow as tf
def _ex(u,m,s):
    f=lambda **k: tf.train.Feature(**k)
    e=tf.train.Example(features=tf.train.Features(feature={
        "user_id":f(bytes_list=tf.train.BytesList(value=[str(u).encode()])),
        "movie_id":f(bytes_list=tf.train.BytesList(value=[str(m).encode()])),
        "movie_title":f(bytes_list=tf.train.BytesList(value=[b""])),
        "segment":f(int64_list=tf.train.Int64List(value=[int(s)])),
        "rating":f(float_list=tf.train.FloatList(value=[0.0]))}))
    return base64.b64encode(e.SerializeToString()).decode()
print("tfx      :", requests.post("http://localhost:8502/v1/models/ranker:predict",
      json={"signature_name":"serving_default","inputs":{"examples":[{"b64":_ex(1,1,0)}]}}, timeout=20).text[:200])

## 6. Launch the dashboard + public URL

In [ ]:
import subprocess, time
subprocess.Popen(["streamlit","run","app/streamsense_explorer.py",
                  "--server.port","8500","--server.headless","true"])
time.sleep(8)
print("If localtunnel shows a password page, the password is this IP:")
!curl -s ifconfig.me; print()
print("Open the https URL printed below:")
!npx --yes localtunnel --port 8500

## 7. Using the demo

Sidebar: switch **Testing** (single TFX ranker) vs **Prod** (retrieval → PyTorch ranker).
Tab 1: pick a user → live recommendations. Tab 2: pick a movie → similar titles + likely viewers.
Tab 3: the committed evaluation metrics.

Stop: `!torchserve --stop` and restart the runtime. If a smoke test is momentarily empty, the server
is still warming up — re-run that one cell.